In [1]:
import pandas as pd

import re
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

In [2]:
df = pd.read_csv('dataset.csv')
df

,Question,Answer
0,Which learning concept describes Leaky negativ...,LeakyReLU
1,What do we call Classification accuracy metric?,Accuracy
2,What is the name for Transformer input represe...,Embedding
3,Which learning concept describes Activation ce...,Tanh
4,Which component handles Bidirectional recurren...,BiRNN
...,...,...
3295,Which organization or person is associated wit...,Chen
3296,Which organization or person is associated wit...,Microsoft
3297,Which organization or person is associated wit...,Yandex
3298,Which organization or person is associated wit...,OpenAI


In [3]:
df['Question'] = df['Question'].str.lower()
df['Answer'] = df['Answer'].str.lower()

df['Question'] = df['Question'].str.replace(
    r'[\?!."\']', '', regex=True
)

df['Answer'] = df['Answer'].str.replace(
    r'[\?!."\']', '', regex=True
)

df['Question'] = df['Question'].str.replace(
    r'\s+', ' ', regex=True
).str.strip()

df['Answer'] = df['Answer'].str.replace(
    r'\s+', ' ', regex=True
).str.strip()

In [4]:
def tokenize(text):
    return text.split()

print(tokenize(df['Question'][0]))

['which', 'learning', 'concept', 'describes', 'leaky', 'negative', 'relu']


In [5]:
vocab = {
    "<PAD>": 0,
    "<UNK>": 1
}

for _, row in df.iterrows():

    question_tokens = tokenize(row['Question'])
    answer_tokens = tokenize(row['Answer'])

    tokens = question_tokens + answer_tokens

    for token in tokens:

        if token not in vocab:
            vocab[token] = len(vocab)

index_to_word = {
    index: word
    for word, index in vocab.items()
}

print("Vocabulary Size:", len(vocab))

Vocabulary Size: 3157


In [6]:
def text_to_indices(text, vocab):

    tokens = tokenize(text)

    indices = [
        vocab.get(token, vocab["<UNK>"])
        for token in tokens
    ]

    return indices

In [7]:
print(text_to_indices(df['Question'][111], vocab))

[10, 17, 65, 296, 131]


In [8]:
class QADataset(Dataset):

    def __init__(self, df, vocab):

        self.df = df
        self.vocab = vocab

    def __len__(self):

        return len(self.df)

    def __getitem__(self, index):

        question = self.df.iloc[index]['Question']
        answer = self.df.iloc[index]['Answer']

        question_indices = text_to_indices(
            question,
            self.vocab
        )

        answer_indices = text_to_indices(
            answer,
            self.vocab
        )

        return (
            torch.tensor(question_indices),
            torch.tensor(answer_indices)
        )

In [9]:
dataset = QADataset(df, vocab)

dataloader = DataLoader(
    dataset,
    batch_size=1,
    shuffle=True
)

sample_question, sample_answer = dataset[2]

print("Sample Question Indices:", sample_question)
print("Sample Answer Indices:", sample_answer)

Sample Question Indices: tensor([10, 17, 18, 19, 20, 21, 22, 23])
Sample Answer Indices: tensor([24])


In [10]:
class SimpleLSTM(nn.Module):

    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):

        super(SimpleLSTM, self).__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim
        )

        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_dim,
            output_dim
        )

    def forward(self, x):

        embedded = self.embedding(x)

        output, (hidden, cell) = self.lstm(embedded)

        output = self.fc(hidden[-1])

        return output

In [11]:
model = SimpleLSTM(
    vocab_size=len(vocab),
    embedding_dim=60,
    hidden_dim=100,
    output_dim=len(vocab)
)

print(model)

SimpleLSTM(
  (embedding): Embedding(3157, 60)
  (lstm): LSTM(60, 100, batch_first=True)
  (fc): Linear(in_features=100, out_features=3157, bias=True)
)


In [12]:
dummy_input = torch.tensor([
    text_to_indices(df['Question'][0], vocab)
])

dummy_output = model(dummy_input)

print("Dummy Output Shape:", dummy_output.shape)

Dummy Output Shape: torch.Size([1, 3157])


In [13]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [14]:
for epoch in range(10):

    total_loss = 0.0

    for question_indices, answer_indices in dataloader:

        optimizer.zero_grad()

        outputs = model(question_indices)

        answer_indices = answer_indices.squeeze(0)

        loss = criterion(
            outputs,
            answer_indices
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}, "
        f"Loss: {total_loss / len(dataloader)}"
    )

print("Training complete.")

Epoch 1, Loss: 5.8790219536965544
Epoch 2, Loss: 3.0631801346175824
Epoch 3, Loss: 1.786925705882128
Epoch 4, Loss: 1.059445539284921
Epoch 5, Loss: 0.5782101374156562
Epoch 6, Loss: 0.29041266139362987
Epoch 7, Loss: 0.15132470347551244
Epoch 8, Loss: 0.0957360619958446
Epoch 9, Loss: 0.07610504096566288
Epoch 10, Loss: 0.048882935591924304
Training complete.


In [16]:
def predict(question_text,model,vocab,idx_to_word):

    model.eval()

    with torch.no_grad():

        indices = text_to_indices(question_text,vocab)

        if not indices:
            return "<UNK>"

        x = torch.tensor(indices,dtype=torch.long).unsqueeze(0)

        prediction = model(x)

        pred_idx = torch.argmax(prediction,dim=1).item()

        return idx_to_word.get(pred_idx,"<UNK>")

In [17]:
tests = [
    "Which NLP task condenses lengthy text into a concise overview?",

    "In binary classification, what error label denotes a missed positive detection?",

    "Which text preparation step segments sentences into individual units?",

    "Which pooling layer extracts the highest activation value across a feature window?",

    "Which structural property enables an RNN to maintain history across time steps?"
]


for question in tests:

    prediction = predict(
        question,
        model,
        vocab,
        index_to_word
    )

    print("Question:", question)
    print("Predicted:", prediction)
    print("-" * 50)

Question: Which NLP task condenses lengthy text into a concise overview?
Predicted: adagrad
--------------------------------------------------
Question: In binary classification, what error label denotes a missed positive detection?
Predicted: recall
--------------------------------------------------
Question: Which text preparation step segments sentences into individual units?
Predicted: layernorm
--------------------------------------------------
Question: Which pooling layer extracts the highest activation value across a feature window?
Predicted: layernorm
--------------------------------------------------
Question: Which structural property enables an RNN to maintain history across time steps?
Predicted: lstm
--------------------------------------------------


In [18]:
# class SimpleLSTM(nn.Module):

#     def __init__(
#         self,
#         vocab_size,
#         embedding_dim,
#         hidden_dim,
#         output_dim
#     ):

#         super(SimpleLSTM, self).__init__()

#         self.embedding = nn.Embedding(
#             vocab_size,
#             embedding_dim
#         )

#         self.lstm = nn.LSTM(
#             input_size=embedding_dim,
#             hidden_size=hidden_dim,
#             num_layers=2,
#             batch_first=True
#         )

#         self.fc = nn.Linear(
#             hidden_dim,
#             output_dim
#         )

#     def forward(self, x):

#         embedded = self.embedding(x)

#         output, (hidden, cell) = self.lstm(embedded)

#         output = self.fc(hidden[-1])

#         return output